#Load LLM using Ollama

In [ ]:
!sudo apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
#ollama serve & ollama pull Eomer/gpt-3.5-turbo
!ollama pull Eomer/gpt-3.5-turbo

In [ ]:
!ollama serve &

Error: listen tcp 127.0.0.1:11434: bind: address already in use


## LangGraph

In [ ]:
!pip install langchain_openai

In [ ]:
from langgraph.graph import StateGraph
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict

In [ ]:
from langchain_openai import ChatOpenAI

In [ ]:
llm = ChatOpenAI(
    model="Eomer/gpt-3.5-turbo",
    base_url="http://localhost:11434/v1",
    api_key="ollama",
    temperature=0.1
)

In [ ]:
# LangGraph State
class AgentState(TypedDict):
    topic: str
    plan: str
    abstract: str
    introduction: str
    related_work: str
    datasets: str
    methodology: str
    models_used: str
    draft: str
    editor: str
    final: str

In [ ]:
def planner_node(state: AgentState) -> dict:
    prompt = f"You are a researcher who is participating in writing an IEEE research paper,Your role is to Create a detailed outline for the research paper article about: {state['topic']} including: related papers that worked on the same topic, their citations, 2-3 datasets used in this topic"
    response = llm.invoke(prompt)
    print("---PLANNER OUTPUT---")
    print(response.content)
    return {"plan": response.content}

def abstract_node(state: AgentState) -> dict:
    prompt = f"You are a researcher who is participating in writing an IEEE research paper,Your role is to Write the abstract part for a new research paper upon this topic:\n\n{state['topic']}, here is the outline that we will be following:{state['plan']}, summarize it to be an abstract that is talking about what the research paper will do and the models will be used in this paper and write keywords"
    response = llm.invoke(prompt)
    print("---ABSTRACT OUTPUT---")
    print(response.content)
    return {"abstract": response.content}

def related_work_node(state: AgentState) -> dict:
    prompt = f"You are a researcher who is participating in writing an IEEE research paper,Your role is to Write only the summarization part (Litrature review) for these research papers mentioned here in the plan :\n\n{state['plan']} \n\n, Include the: 1. The authors for each paper \n 2. Models with the heighst results \n 3. Each study's conclusion \n 4. Each study's Limitation \n \n Write One paragraph for each paper mentioned"
    response = llm.invoke(prompt)
    print("---RELATED WORK OUTPUT---")
    print(response.content)
    return {"related_work": response.content}

def datasets_node(state: AgentState) -> dict:
    prompt = f"You are a researcher who is participating in writing an IEEE research paper,Your role is to write only about the datasets mentioned in this outline:\n\n{state['plan']} , search for each and descripe in details by showing: 1. Number of features \n 2. Number of records \n 3. Target feature \n 4. Each feature's description. Write one paragraph for each dataset"
    response = llm.invoke(prompt)
    print("---DATASETS OUTPUT---")
    print(response.content)
    return {"datasets": response.content}

def methodology_node(state: AgentState) -> dict:
    prompt = f"You are a researcher who is participating in writing an IEEE research paper,Your role is to Write only the 'Methodolgy' part using this outline: \n{state['plan']} \n\n You must include: 1. (3-5) Models that will be used \n 2. Steps for data preprocessing \n 3. Performance metrics that will measure the accuracy of the models"
    response = llm.invoke(prompt)
    print("---METHODOLOGY OUTPUT---")
    print(response.content)
    return {"methodology": response.content}

def models_used_node(state: AgentState) -> dict:
    prompt = f"You are a researcher who is participating in writing an IEEE research paper,Your role is to Write only the description of the models that will be used. \n Use the models listed here inside the methodlogy planned :{state['methodology']}, search and explain each model seperatly in one paragraph"
    response = llm.invoke(prompt)
    print("---MODELS USED OUTPUT---")
    print(response.content)
    return {"models_used": response.content}

def draft_node(state: AgentState) -> dict:
    prompt = f"You are a researcher who is writing a research paper following the IEEE format, Using these inputs given from other partners, Write the whole research paper with this order given : Abstract , Introduction, Related work, datasets used, methodolgy, models used. \n \n Don't add or edit anything, Just use the format and append each part using the IEEE format, here are the parts in order: Abstract :{state['abstract']} \n \n Related work:  {state['related_work']}\n \n Datasets: {state['datasets']}\n \n Methodolgy: {state['methodology']}\n \n Models Used: {state['models_used']}"
    response = llm.invoke(prompt)
    print("---DRAFT OUTPUT---")
    print(response.content)
    return {"draft": response.content}

def editor_node(state: AgentState) -> dict:
    prompt = f"You are an IEEE researcher who is reviewing a research paper, editing and refining it. Fix grammar mistakes and clarity while following the IEEE research paper format, here is the draft :\n\n{state['draft']},Explain what to edit in the draft (if needed) to finalize it to be published and send the review"
    response = llm.invoke(prompt)
    print("---EDITOR OUTPUT---")
    print(response.content)
    return {"editor": response.content}

def finalize_node(state: AgentState) -> dict:
    prompt = f"You are an IEEE researcher whose role is to re-write a draft using a reviewer's report ,follow the reviewer's refinement and instructions and rewrite the whole research paper correctly with the same draft. Here is the draft: {state['draft']} \n \n and here is the reviewr's report for editing :{state['editor']}, Rewrite the research paper correctly"
    response = llm.invoke(prompt)
    print("---FINALIZE OUTPUT---")
    print(response.content)
    return {"final": response.content}

In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(AgentState)

# Add nodes
builder.add_node("planner", planner_node)
builder.add_node("abstract", abstract_node)
builder.add_node("related_work", related_work_node)
builder.add_node("datasets", datasets_node)
builder.add_node("methodology", methodology_node)
builder.add_node("models_used", models_used_node)
builder.add_node("writer", draft_node)
builder.add_node("editor", editor_node)
builder.add_node("final", finalize_node)

# Define the sequential flow
builder.add_edge(START, "planner")
builder.add_edge("planner", "abstract")
builder.add_edge("abstract", "related_work")
builder.add_edge("related_work", "datasets")
builder.add_edge("datasets", "methodology")
builder.add_edge("methodology", "models_used")
builder.add_edge("models_used", "writer")
builder.add_edge("writer", "editor")
builder.add_edge("editor", "final")
builder.add_edge("final", END)

# Compile
graph = builder.compile()

In [ ]:
pip install python-telegram-bot --quiet

In [ ]:
!pip install fpdf

In [ ]:
from fpdf import FPDF

def save_as_pdf_with_header(header_text: str, content: str, output_filename: str):
    """
    Save text content as a PDF with a persistent page header.

    Args:
        header_text: The text to display as the header (appears on every page).
        content: The main body text.
        output_filename: The name of the output PDF file (e.g., "report.pdf").
    """
    class PDF(FPDF):
        def header(self):
            self.set_font("Arial", "B", 12)
            self.cell(0, 10, header_text, align="C", ln=True)
            self.line(10, self.get_y(), 200, self.get_y())  # thin separator
            self.ln(5)

    pdf = PDF()
    pdf.add_page()
    pdf.set_font("Arial", size=12)
    pdf.multi_cell(0, 10, content)
    pdf.output(output_filename)
    print(f"PDF saved as {output_filename}")

In [ ]:
from telegram import Update
from telegram.ext import Application, CommandHandler, MessageHandler, filters, ContextTypes

TOKEN = ''

chat_history = []


In [ ]:
async def start(update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
    """Sends a message when the command /start is issued."""
    user = update.effective_user
    await update.message.reply_html(
        f"Hi {user.mention_html()}! I'm a multiagent that wil generate to you a PDF output for a specific research topic",
    )


In [ ]:
async def echo(update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
    """Processes user input as a topic, runs the LangGraph, saves as PDF, and sends it back."""
    topic = update.message.text
    await update.message.reply_text(f"Processing your request for topic: '{topic}'... This might take a moment.")

    try:
        # Run the LangGraph with the user-provided topic
        final_state = graph.invoke(
            {"topic": topic},
            config={"configurable": {"thread_id": "session-1"}}
        )

        # Extract content and header
        pdf_content = final_state.get("final", "No final content generated.")
        pdf_header = final_state.get("topic", "Untitled Article")
        output_filename = f"{pdf_header.replace(' ', '_').replace('/', '_')}.pdf"

        # Save as PDF
        save_as_pdf_with_header(
            header_text=pdf_header,
            content=pdf_content,
            output_filename=output_filename
        )

        # Send the PDF back to the user
        with open(output_filename, 'rb') as f:
            await update.message.reply_document(document=f, filename=output_filename)

        await update.message.reply_text("Here is your generated research paper!")

    except Exception as e:
        await update.message.reply_text(f"An error occurred: {e}")

In [ ]:
import nest_asyncio
nest_asyncio.apply()

def main() -> None:
    """Start the bot."""
    # Create the Application and pass it your bot's token.
    application = Application.builder().token(TOKEN).build()

    # on different commands - answer in Telegram
    application.add_handler(CommandHandler("start", start))
    application.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, echo))

    # Run the bot until the user presses Ctrl-C
    print("Bot started. Send /start to the bot to begin.")
    application.run_polling(allowed_updates=Update.ALL_TYPES)

if __name__ == '__main__':
    main()

Bot started. Send /start to the bot to begin.
---PLANNER OUTPUT---

Great! As a friendly assistant, I'd be happy to help you create an outline for your IEEE research paper on predictive modeling of Bitcoin prices using time-series forecasting models. Here's a detailed outline that covers the key elements you'll need to include in your paper:

I. Introduction
A. Background and motivation
- Briefly explain the importance of predicting Bitcoin prices, especially for investors and traders.
- Discuss the challenges of predicting cryptocurrency prices and the potential benefits of using time-series forecasting models.
B. Research question and objectives
- Clearly state the research question you are trying to answer.
- Explain the objectives of your study and how they align with the research question.
C. Organization of the paper
- Provide an overview of the structure of your paper, including the sections that will follow.

II. Literature Review
A. Overview of existing work on Bitcoin price p

RuntimeError: Cannot close a running event loop